# LeetCode 4059 — Lexicographically Largest Power Array (Hard)

Bài này đến từ **Weekly Contest 520 (câu 4)**, nhãn chủ đề: *Array, Bit Manipulation, Greedy*.

Skeleton dạng LeetCode (Python 3):

```python
class Solution:
    def lexLargestPower(self, nums: List[int]) -> List[int]:
        ...
```

> **Lưu ý về tên method:** bản đề mình có (mirror leetcode.doocs.org) **không ghi** tên method. Ở đây mình đặt là `lexLargestPower`. Khi nộp bài, bạn chỉ cần **giữ nguyên thân hàm** và đổi tên method theo đúng skeleton LeetCode hiển thị. Kiểu tham số `nums: List[int]` và kiểu trả về `List[int]` thì chắc chắn đúng theo đề (vào một mảng số, ra mảng `power` độ dài 15).

Bài này nghe có vẻ "đáng sợ" (Hard, bit, lexicographic, hoán vị...). Nhưng nếu đi từ từ, bạn sẽ thấy nó chỉ là **một hình ảnh rất đơn giản**: xếp các số thành hàng dưới dạng nhị phân, rồi đếm số chữ `1` liên tiếp từ trên xuống ở mỗi cột.

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc **đúng** định nghĩa `power`: chỉ số `i` ứng với **bit `14 - i`**, và "`j` phần tử đầu tiên đều có bit".
2. Các thao tác bit trong Python: `bin`, `format(x, '015b')`, `>>`, `&`, `1 << b`, và **AND tiền tố (prefix AND)**.
3. **So sánh theo thứ tự từ điển (lexicographic)** là gì, và Python so sánh hai `list` như thế nào.
4. Viết **brute force** bằng `itertools.permutations` để có "đáp án chuẩn" dùng cho kiểm thử.
5. Insight "AND của một tiền tố chỉ phụ thuộc **tập** phần tử, không phụ thuộc thứ tự" → **DP bitmask** + mẹo **mã hoá vector thành một số cơ số `n + 1`**.
6. Kỹ thuật chính: **Greedy theo thứ tự ưu tiên** + **chia nhóm có thứ tự** (*ordered partition refinement*), kèm chứng minh bằng invariant.
7. Tối ưu cài đặt bằng `Counter` + "AND của cả nhóm", kiểm thử chéo (cross-check), và các lỗi người mới hay mắc.

## Các cách giải sẽ học (từ dễ đến tối ưu)

| Cách | Ý tưởng | Độ phức tạp thời gian |
|---|---|---|
| Cách 1 | Brute force: thử mọi hoán vị, lấy `power` lớn nhất | `O(n! · 15 · n)` |
| Cách 2 | DP bitmask trên tập con (dùng insight "tập, không phải thứ tự") | `O(2^n · n)` |
| Cách 3 | Greedy theo bit từ cao xuống thấp + chia nhóm có thứ tự | `O(15 · n)` |
| Cách 4 | Cùng greedy, nhưng nén bằng `Counter` + lưu AND của mỗi nhóm | `O(n + 15 · D)` |

(`D` = số giá trị **khác nhau** trong `nums`, `D ≤ min(n, 2^15)`.)

# 1. Đề bài (dịch sát nghĩa)

Cho một mảng số nguyên `nums` có độ dài `n`. Bạn được phép **sắp xếp lại** các phần tử để tạo thành **một hoán vị bất kỳ** `perm` (permutation — một cách xếp lại thứ tự, giữ nguyên các giá trị).

Định nghĩa mảng `power` có **độ dài 15**. Với mỗi `0 <= i < 15`:

> `power[i]` là số nguyên `j` **lớn nhất**, với `0 <= j <= n`, sao cho **`j` phần tử đầu tiên** của `perm` **đều** có bit thứ `(14 - i)` được bật (bằng `1`).

Vị trí bit được đánh số **từ phải sang trái**, bắt đầu từ bit thứ `0`.

Hãy trả về mảng `power` **lớn nhất theo thứ tự từ điển** (*lexicographically largest*) có thể đạt được.

---

## Ví dụ 1

```text
Input:  nums = [7, 5]
Output: [0,0,0,0,0,0,0,0,0,0,0,0,2,1,2]
```

Chọn `perm = [7, 5]`:

- Cả hai phần tử đều có bit 2 → `power[12] = 2`.
- Phần tử đầu có bit 1, phần tử thứ hai thì không → `power[13] = 1`.
- Cả hai phần tử đều có bit 0 → `power[14] = 2`.
- Các bit cao hơn đều tắt ở phần tử đầu, nên các ô còn lại bằng `0`.

## Ví dụ 2

```text
Input:  nums = [3, 1, 7]
Output: [0,0,0,0,0,0,0,0,0,0,0,0,1,2,3]
```

Chọn `perm = [7, 3, 1]`:

- Phần tử đầu có bit 2, phần tử thứ hai thì không → `power[12] = 1`.
- Hai phần tử đầu có bit 1, phần tử thứ ba thì không → `power[13] = 2`.
- Cả ba phần tử đều có bit 0 → `power[14] = 3`.
- Các bit cao hơn đều tắt ở phần tử đầu, nên các ô còn lại bằng `0`.

## Ràng buộc (constraints)

- `1 <= nums.length <= 5 * 10^4`
- `0 <= nums[i] < 2^15`

## 1.1. Đọc kỹ định nghĩa — ba chỗ dễ hiểu sai

**(a) Chỉ số `i` của `power` chạy ngược với chỉ số bit.**

| `i` (vị trí trong `power`) | 0 | 1 | ... | 12 | 13 | 14 |
|---|---|---|---|---|---|---|
| bit được xét `b = 14 - i` | 14 | 13 | ... | 2 | 1 | 0 |

Vậy `power[0]` nói về **bit cao nhất** (bit 14), còn `power[14]` nói về **bit 0** (bit thấp nhất, bit "lẻ/chẵn").

**(b) "`j` phần tử ĐẦU TIÊN đều có bit"** — tức là đếm một **chuỗi liên tiếp từ đầu mảng**. Gặp phần tử đầu tiên không có bit đó là dừng, những phần tử phía sau dù có bit cũng **không** được tính. `j = 0` luôn hợp lệ (0 phần tử thì "đều có" một cách hiển nhiên).

**(c) "Lớn nhất theo thứ tự từ điển"**: so sánh `power[0]` trước; chỉ khi bằng nhau mới xét `power[1]`; và cứ thế. Một vector có `power[0]` lớn hơn **luôn** thắng, bất kể các ô sau ra sao.

Hệ quả quan trọng của (a) + (c): **bit cao quan trọng hơn bit thấp**. Ta ưu tiên làm bit 14 "sống lâu" nhất, rồi mới đến bit 13, ...

In [1]:
from typing import List

examples = [
    ([7, 5], [0,0,0,0,0,0,0,0,0,0,0,0,2,1,2]),
    ([3, 1, 7], [0,0,0,0,0,0,0,0,0,0,0,0,1,2,3]),
]

for nums, expected in examples:
    print("nums     =", nums, "| n =", len(nums))
    for x in nums:
        print(f"   {x:>2} -> nhị phân 15 bit: {format(x, '015b')}")
    print("expected =", expected)
    print()

print("Bảng ánh xạ i -> bit 14 - i:")
print([(i, 14 - i) for i in range(15)])

nums     = [7, 5] | n = 2
    7 -> nhị phân 15 bit: 000000000000111
    5 -> nhị phân 15 bit: 000000000000101
expected = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 2]

nums     = [3, 1, 7] | n = 3
    3 -> nhị phân 15 bit: 000000000000011
    1 -> nhị phân 15 bit: 000000000000001
    7 -> nhị phân 15 bit: 000000000000111
expected = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3]

Bảng ánh xạ i -> bit 14 - i:
[(0, 14), (1, 13), (2, 12), (3, 11), (4, 10), (5, 9), (6, 8), (7, 7), (8, 6), (9, 5), (10, 4), (11, 3), (12, 2), (13, 1), (14, 0)]


## 1.2. Constraints nói gì về độ phức tạp?

- `n` có thể tới `5 · 10^4`.
  - Thử mọi hoán vị: `n!` — với `n = 20` đã là `2.4 · 10^18`. **Không thể.**
  - Thử mọi tập con: `2^n` — với `n = 50` đã là `10^15`. **Không thể.**
  - Ta cần cỡ `O(n log n)` hoặc `O(15 · n)` ≈ `7.5 · 10^5` bước — Python chạy trong tích tắc.
- `nums[i] < 2^15` nghĩa là mỗi số có **đúng 15 bit** (bit 0 … bit 14). Đó là lý do `power` có độ dài 15: mỗi ô ứng với một bit. Không có số nào có bit 15 trở lên.
- Số bit là **hằng số nhỏ (15)** → các thuật toán kiểu "duyệt từng bit, mỗi bit duyệt mảng một lần" là hợp lý.

> **Dấu hiệu nhận dạng:** "giá trị < 2^15" + "mảng dài 5 · 10^4" thường gợi ý: *vòng lặp ngoài theo bit (15 lần), vòng trong theo phần tử*.

# 2. Hiểu đề bằng tay trước khi nghĩ thuật toán

## 2.1. Hình ảnh then chốt: xếp các số thành "lưới bit"

Viết mỗi phần tử của `perm` thành **một hàng** gồm 15 ký tự nhị phân (bit 14 ở bên trái, bit 0 ở bên phải). Xếp các hàng chồng lên nhau theo thứ tự của `perm`.

Khi đó **ký tự thứ `i` của chuỗi** (đếm từ trái, bắt đầu từ 0) chính là **bit `14 - i`** — trùng khớp với ô `power[i]`!

> `power[i]` = số chữ `1` **liên tiếp từ trên xuống** ở **cột `i`** của lưới.

Ví dụ 1 với `perm = [7, 5]` (chỉ vẽ 3 cột cuối, các cột trước toàn `0`):

```text
cột i:     12 13 14
bit b:      2  1  0
  7   ->    1  1  1
  5   ->    1  0  1
power:      2  1  2      (cột 13: 1 rồi 0 -> dừng ở 1)
```

Mọi cột từ 0 đến 11: hàng đầu tiên đã là `0` → `power = 0`.

Cell dưới đây vẽ lưới này bằng **chuỗi** (chưa cần phép toán bit nào) để bạn nhìn tận mắt.

In [2]:
def show_grid(perm):
    """Vẽ lưới bit của perm và tính power bằng cách đếm '1' liên tiếp từ trên xuống mỗi cột."""
    rows = [format(x, '015b') for x in perm]          # mỗi số -> chuỗi 15 ký tự
    print("   cột i: " + "".join(str(i % 10) for i in range(15)))
    for x, r in zip(perm, rows):
        print(f"{x:>8} | {r}")
    power = []
    for i in range(15):                                 # cột i
        j = 0
        while j < len(rows) and rows[j][i] == '1':      # đi xuống khi còn gặp '1'
            j += 1
        power.append(j)
    print("power =", power)
    return power

show_grid([7, 5])
print()
show_grid([7, 3, 1]);

   cột i: 012345678901234
       7 | 000000000000111
       5 | 000000000000101
power = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 2]

   cột i: 012345678901234
       7 | 000000000000111
       3 | 000000000000011
       1 | 000000000000001
power = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3]


## 2.2. Thử tay tất cả hoán vị của `nums = [9, 6, 3]`

Nhị phân (4 bit cuối): `9 = 1001`, `6 = 0110`, `3 = 0011`. Các ô `power[0..10]` luôn bằng 0 (không số nào có bit ≥ 4), nên ta chỉ ghi 4 ô cuối `power[11..14]` (ứng với bit 3, 2, 1, 0).

| `perm` | bit 3 (`power[11]`) | bit 2 (`power[12]`) | bit 1 (`power[13]`) | bit 0 (`power[14]`) |
|---|:---:|:---:|:---:|:---:|
| `[9, 6, 3]` | 1 | 0 | 0 | 1 |
| `[9, 3, 6]` | 1 | 0 | 0 | **2** |
| `[6, 9, 3]` | 0 | 1 | 1 | 0 |
| `[6, 3, 9]` | 0 | 1 | 2 | 0 |
| `[3, 9, 6]` | 0 | 0 | 1 | 2 |
| `[3, 6, 9]` | 0 | 0 | 2 | 1 |

Cách đọc một dòng, ví dụ `[9, 3, 6]`:

- bit 3: `9` có (1001) → tiếp `3` không có (0011) → dừng: **1**.
- bit 2: `9` không có → dừng ngay: **0**.
- bit 1: `9` không có → **0**.
- bit 0: `9` có, `3` có, `6` không có (0110) → **2**.

So sánh từ điển: các dòng bắt đầu bằng `0` ở bit 3 thua hết. Giữa `[9,6,3] → 1,0,0,1` và `[9,3,6] → 1,0,0,2`, ô cuối quyết định: **`[9, 3, 6]` thắng**.

Đáp án: `power = [0,0,0,0,0,0,0,0,0,0,0,1,0,0,2]`.

Chú ý: dòng `[6, 3, 9]` có ô bit 1 bằng **2** (lớn hơn ô bit 1 bằng 0 của `[9, 3, 6]`), và tổng các ô của nó cũng bằng 3 như `[9, 3, 6]` — nhưng nó vẫn thua, vì ngay ô đầu tiên khác nhau (bit 3) nó chỉ có 0. Lexicographic không quan tâm tổng hay các ô phía sau — chỉ quan tâm **ô đầu tiên khác nhau**.

In [3]:
from itertools import permutations

for perm in permutations([9, 6, 3]):
    rows = [format(x, '015b') for x in perm]
    last4 = []
    for i in range(11, 15):
        j = 0
        while j < len(rows) and rows[j][i] == '1':
            j += 1
        last4.append(j)
    print(list(perm), "-> power[11..14] =", last4)

[9, 6, 3] -> power[11..14] = [1, 0, 0, 1]
[9, 3, 6] -> power[11..14] = [1, 0, 0, 2]
[6, 9, 3] -> power[11..14] = [0, 1, 1, 0]
[6, 3, 9] -> power[11..14] = [0, 1, 2, 0]
[3, 9, 6] -> power[11..14] = [0, 0, 1, 2]
[3, 6, 9] -> power[11..14] = [0, 0, 2, 1]


## 2.3. Ba nhận xét rút ra từ việc làm tay

1. **Hình ảnh cột:** `power[i]` = độ dài "cột chữ 1" bắt đầu từ hàng trên cùng ở cột `i`. Một cột bị "gãy" ở hàng nào thì các hàng phía dưới không còn ý nghĩa với cột đó.
2. **Bit cao được ưu tiên tuyệt đối:** muốn thắng, trước hết phải làm cột bit cao nhất (có ai đó có bit ấy) dài nhất có thể — ở `[9,6,3]` là bit 3, nên `9` **bắt buộc** đứng đầu.
3. **Xếp "giảm dần" không phải lúc nào cũng tốt:** `[9, 6, 3]` (giảm dần) chỉ được `1,0,0,1`, thua `[9, 3, 6]`. Sau khi `9` đứng đầu, việc chọn ai tiếp theo phụ thuộc vào **bit nào còn "sống"** — ở đây chỉ bit 0 còn cơ hội, nên ta ưu tiên `3` (có bit 0) chứ không phải `6` (lớn hơn).

Nhận xét 3 là một **cái bẫy** rất phổ biến; ta sẽ quay lại nó ở phần "Bước đột phá" và "Lỗi thường gặp".

# 3. Nền tảng (học trước những công cụ sẽ dùng)

## 3.1. Hệ nhị phân và vị trí bit

Mỗi số nguyên không âm viết được dưới dạng tổng các lũy thừa của 2:

```text
11 = 8 + 2 + 1 = 1·2^3 + 0·2^2 + 1·2^1 + 1·2^0  ->  1011
```

- **Bit thứ `b`** là hệ số đứng trước `2^b`. Đếm **từ phải sang trái**, bắt đầu từ 0.
- Với `11 = 1011`: bit 0 = 1, bit 1 = 1, bit 2 = 0, bit 3 = 1.

Trong Python:

- `bin(11)` → `'0b1011'` (có tiền tố `0b`).
- `format(11, '015b')` → chuỗi **15 ký tự**, đệm số `0` bên trái: `'000000000001011'`.
  - `'b'` = binary, `'15'` = độ rộng 15, `'0'` = đệm bằng số 0.
- Với chuỗi 15 ký tự này, **ký tự ở vị trí `i` là bit `14 - i`**. Đây chính là phép ánh xạ `power[i] ↔ bit 14 - i` của đề.

In [4]:
x = 11
s = format(x, '015b')
print("bin(11)            =", bin(x))
print("format(11,'015b')  =", s, "| độ dài", len(s))

for i in [11, 12, 13, 14]:
    print(f"ký tự s[{i}] = {s[i]}  <->  bit {14 - i}")

print("Số lớn nhất được phép: 2**15 - 1 =", 2**15 - 1, "->", format(2**15 - 1, '015b'))

bin(11)            = 0b1011
format(11,'015b')  = 000000000001011 | độ dài 15
ký tự s[11] = 1  <->  bit 3
ký tự s[12] = 0  <->  bit 2
ký tự s[13] = 1  <->  bit 1
ký tự s[14] = 1  <->  bit 0
Số lớn nhất được phép: 2**15 - 1 = 32767 -> 111111111111111


## 3.2. Toán tử bit: `>>`, `<<`, `&`

| Toán tử | Đọc là | Ví dụ | Kết quả |
|---|---|---|---|
| `x >> b` | dịch phải `b` bit (bỏ `b` bit cuối) | `11 >> 1` (1011 → 101) | `5` |
| `1 << b` | dịch trái: số chỉ có bit `b` bật | `1 << 3` | `8` (1000) |
| `x & y` | AND từng bit: bit = 1 khi **cả hai** là 1 | `11 & 6` (1011 & 0110) | `2` (0010) |

**Kiểm tra bit `b` của `x` có bật không** — hai cách tương đương:

```python
(x >> b) & 1        # ra 0 hoặc 1
x & (1 << b) != 0   # ra True/False
```

> **Cẩn thận độ ưu tiên toán tử!** Trong Python, phép so sánh (`==`, `!=`) có độ ưu tiên **thấp hơn** `&`. Viết `x & (1 << b) == 1` sẽ được hiểu là `(x & (1 << b)) == 1` — chỉ đúng khi `b = 0`! (Xem phần lỗi thường gặp.) Mẹo: **luôn thêm ngoặc**.

In [5]:
x = 11  # 1011
print("11 >> 1 =", 11 >> 1, "| 1 << 3 =", 1 << 3, "| 11 & 6 =", 11 & 6)

for b in range(4):
    print(f"bit {b} của {x}: (x >> b) & 1 = {(x >> b) & 1} | x & (1 << b) != 0 -> {x & (1 << b) != 0}")

11 >> 1 = 5 | 1 << 3 = 8 | 11 & 6 = 2
bit 0 của 11: (x >> b) & 1 = 1 | x & (1 << b) != 0 -> True
bit 1 của 11: (x >> b) & 1 = 1 | x & (1 << b) != 0 -> True
bit 2 của 11: (x >> b) & 1 = 0 | x & (1 << b) != 0 -> False
bit 3 của 11: (x >> b) & 1 = 1 | x & (1 << b) != 0 -> True


## 3.3. AND của nhiều số và "prefix AND"

AND nhiều số: `a & b & c` có bit `b` bật **khi và chỉ khi mọi số** đều có bit `b`.

Vậy câu "**`j` phần tử đầu tiên đều có bit `b`**" tương đương với:

> bit `b` bật trong `perm[0] & perm[1] & ... & perm[j-1]`.

Gọi `A_k` là AND của `k` phần tử đầu (**prefix AND**). Tính chất quan trọng:

- AND **chỉ có thể làm tắt bit, không bao giờ bật thêm**. Nên `A_1 ⊇ A_2 ⊇ A_3 ⊇ ...` (theo tập bit), tức dãy prefix AND **không tăng**.
- Một khi bit `b` tắt trong `A_k`, nó tắt mãi mãi trong `A_{k+1}, A_{k+2}, ...`.

Hệ quả: `power[i]` = **số lượng** `k` (từ 1 đến `n`) sao cho bit `14 - i` còn bật trong `A_k`.

Quy ước "AND của 0 phần tử" là số có **mọi bit bật**: `(1 << 15) - 1 = 32767` (phần tử trung hòa của AND, giống như 0 với phép cộng hay 1 với phép nhân).

In [6]:
perm = [7, 3, 1]
acc = (1 << 15) - 1                 # AND của 0 phần tử: mọi bit đều bật
print("ban đầu     acc =", format(acc, '015b'))
for k, x in enumerate(perm, start=1):
    acc &= x                        # acc = acc & x
    print(f"k={k}, x={x}: A_{k} = {format(acc, '015b')}  (= {acc})")

ban đầu     acc = 111111111111111
k=1, x=7: A_1 = 000000000000111  (= 7)
k=2, x=3: A_2 = 000000000000011  (= 3)
k=3, x=1: A_3 = 000000000000001  (= 1)


## 3.4. So sánh theo thứ tự từ điển (lexicographic) và `list` trong Python

Thứ tự từ điển là cách ta xếp từ trong từ điển: so chữ cái đầu; nếu bằng thì so chữ thứ hai; ...

Với hai mảng `a`, `b` cùng độ dài: tìm **vị trí đầu tiên `k` mà `a[k] != b[k]`**; mảng nào có giá trị lớn hơn ở `k` thì lớn hơn.

Python so sánh `list` **đúng theo cách này**, nên ta có thể dùng trực tiếp `>`, `<`, `max(...)`:

```python
[1, 0, 0, 2] > [1, 0, 0, 1]   # True  (khác ở vị trí cuối)
[1, 0, 0, 0] > [0, 9, 9, 9]   # True  (khác ngay vị trí đầu, 1 > 0)
```

In [7]:
print([1, 0, 0, 2] > [1, 0, 0, 1])
print([1, 0, 0, 0] > [0, 9, 9, 9])
print(max([[0, 1, 2, 0], [1, 0, 0, 1], [1, 0, 0, 2], [0, 0, 2, 1]]))

True
True
[1, 0, 0, 2]


## 3.5. `itertools.permutations` và tốc độ tăng của `n!`

`permutations(nums)` sinh lần lượt **mọi cách xếp** các phần tử của `nums` (mỗi lần là một `tuple`). Với `n` phần tử có `n! = 1 · 2 · ... · n` cách.

Lưu ý: nếu `nums` có phần tử trùng giá trị, `permutations` vẫn coi chúng là khác nhau (theo vị trí), nên có thể sinh các hoán vị trùng lặp — không sai, chỉ tốn thời gian.

In [8]:
from itertools import permutations
from math import factorial

print(list(permutations([1, 2, 3])))
for n in [3, 8, 10, 15, 20]:
    print(f"n = {n:>2}: n! = {factorial(n):,}")

[(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]
n =  3: n! = 6
n =  8: n! = 40,320
n = 10: n! = 3,628,800
n = 15: n! = 1,307,674,368,000
n = 20: n! = 2,432,902,008,176,640,000


## 3.6. Bitmask: biểu diễn một **tập con** bằng một số nguyên

Với `n` phần tử đánh số `0..n-1`, một tập con `S` được mã hoá bằng số nguyên mà **bit `i` bật ⇔ phần tử `i` thuộc `S`**.

| Thao tác | Code |
|---|---|
| `i` có thuộc `S`? | `(S >> i) & 1` |
| thêm `i` vào `S` | `S \| (1 << i)` |
| bỏ `i` khỏi `S` | `S ^ (1 << i)` (khi `i ∈ S`) |
| bit thấp nhất của `S` | `S & -S` |
| chỉ số của bit đó | `(S & -S).bit_length() - 1` |
| tập đầy đủ | `(1 << n) - 1` |

Một tính chất rất hữu ích cho DP: nếu `T` là tập con **thật sự** của `S` thì `T < S` (dưới dạng số). Do đó duyệt `S = 0, 1, 2, ...` theo thứ tự tăng thì mọi tập con nhỏ hơn đã được xử lý trước.

In [9]:
n = 3
S = 0b101                      # tập {0, 2}
print("S =", format(S, '03b'), "-> các phần tử:", [i for i in range(n) if (S >> i) & 1])
print("thêm 1  ->", format(S | (1 << 1), '03b'))
print("bỏ 2    ->", format(S ^ (1 << 2), '03b'))
low = S & -S
print("bit thấp nhất:", low, "-> chỉ số", low.bit_length() - 1)
print("tập đầy đủ:", format((1 << n) - 1, '03b'))

S = 101 -> các phần tử: [0, 2]
thêm 1  -> 111
bỏ 2    -> 001
bit thấp nhất: 1 -> chỉ số 0
tập đầy đủ: 111


## 3.7. Mẹo: mã hoá một vector thành **một số** để so sánh lexicographic

Giả sử mọi ô của vector đều nằm trong `[0, n]`. Xem vector như các **chữ số** của một số viết ở **cơ số `n + 1`** (ô đầu là chữ số có trọng số lớn nhất):

```text
value = v[0]·(n+1)^(L-1) + v[1]·(n+1)^(L-2) + ... + v[L-1]·(n+1)^0
```

Vì mỗi "chữ số" nhỏ hơn cơ số, **so sánh hai số = so sánh hai vector theo từ điển** (giống so sánh hai số thập phân cùng số chữ số).

Ví dụ `n = 3` (cơ số 4), vector 4 ô:

- `[1, 0, 0, 2]` → `1·64 + 0·16 + 0·4 + 2 = 66`
- `[1, 0, 1, 0]` → `64 + 4 = 68`
- `[1, 0, 0, 1]` → `65`

Thứ tự `68 > 66 > 65` đúng bằng thứ tự từ điển. Lợi ích: một vector "lexicographic" trở thành **một số có thể cộng dồn** — chìa khoá cho Cách 2.

In [10]:
def encode(vec, base):
    value = 0
    for d in vec:                  # giống đọc số thập phân: value = value*10 + chữ số
        value = value * base + d
    return value

def decode(value, base, length):
    vec = []
    for _ in range(length):
        vec.append(value % base)
        value //= base
    return vec[::-1]

for v in ([1, 0, 0, 2], [1, 0, 1, 0], [1, 0, 0, 1]):
    print(v, "->", encode(v, 4), "-> decode:", decode(encode(v, 4), 4, 4))

[1, 0, 0, 2] -> 66 -> decode: [1, 0, 0, 2]
[1, 0, 1, 0] -> 68 -> decode: [1, 0, 1, 0]
[1, 0, 0, 1] -> 65 -> decode: [1, 0, 0, 1]


## 3.8. `Counter` và phép gán lát cắt `groups[t:t+1] = [...]`

**`Counter`** (trong `collections`) đếm tần suất: `Counter([5, 5, 7])` → `{5: 2, 7: 1}`. Dùng ở Cách 4 để gộp các giá trị trùng nhau.

**Gán lát cắt (slice assignment):** `groups[t:t+1] = [A, B]` thay **một** phần tử ở vị trí `t` bằng **hai** phần tử `A, B` (danh sách dài thêm 1). Đây là thao tác "**tách nhóm** `t` thành hai nhóm" mà Cách 3 dùng.

In [11]:
from collections import Counter

cnt = Counter([5, 5, 7, 5])
print(cnt, "| số lần xuất hiện của 5:", cnt[5], "| tổng:", sum(cnt.values()))

groups = [[11], [3, 6, 2, 7], [99]]
t = 1
groups[t:t+1] = [[3, 7], [6, 2]]     # tách nhóm 1 thành hai nhóm
print(groups)

Counter({5: 3, 7: 1}) | số lần xuất hiện của 5: 3 | tổng: 4
[[11], [3, 7], [6, 2], [99]]


# 4. Cách 1 — Brute force: thử mọi hoán vị

## Ý tưởng

Đề cho ta **tự do chọn** `perm`. Cách thẳng thắn nhất:

1. Sinh mọi hoán vị `perm` của `nums`.
2. Với mỗi `perm`, tính `power` **đúng theo định nghĩa**: với mỗi `i`, đi từ đầu mảng tới khi gặp phần tử không có bit `14 - i`.
3. Giữ lại `power` lớn nhất (Python so sánh `list` theo từ điển nên chỉ cần `>`).

Hàm `power_of(perm)` dưới đây sẽ được **dùng lại rất nhiều lần** để kiểm chứng các cách sau. Nó làm đúng việc `show_grid` ở mục 2 làm, nhưng dùng phép toán bit thay cho chuỗi.

In [12]:
from itertools import permutations
from typing import List

BITS = 15

def power_of(perm) -> List[int]:
    """Tính mảng power của một hoán vị cố định, đúng theo định nghĩa của đề."""
    n = len(perm)
    power = [0] * BITS
    for i in range(BITS):
        b = 14 - i                                     # ô i nói về bit 14 - i
        j = 0
        while j < n and ((perm[j] >> b) & 1) == 1:     # j phần tử đầu đều có bit b
            j += 1
        power[i] = j
    return power


def largest_power_bruteforce(nums: List[int]) -> List[int]:
    best = None
    for perm in permutations(nums):
        cur = power_of(perm)
        if best is None or cur > best:                 # so sánh list = so sánh lexicographic
            best = cur
    return best


print(largest_power_bruteforce([7, 5]))
print(largest_power_bruteforce([3, 1, 7]))
print(largest_power_bruteforce([9, 6, 3]))
print(largest_power_bruteforce([3, 11, 6, 2, 7]))

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 2]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 5, 3]


## Brute force tốn bao nhiêu công?

- Số hoán vị: `n!`.
- Mỗi hoán vị: tính `power` mất tối đa `15 · n` bước; so sánh hai `list` mất `15` bước.
- Tổng: `O(n! · 15 · n)`.

Cell dưới đếm thực tế với `n = 7` (5040 hoán vị) và ước lượng cho các `n` lớn hơn. Với `n = 5 · 10^4`, con số `n!` có hơn 200 nghìn chữ số — **không có máy nào chạy nổi**.

Nhưng brute force vẫn **rất quý**: nó đúng hiển nhiên theo đề, nên là "trọng tài" để kiểm tra mọi cách thông minh hơn trên input nhỏ.

In [13]:
import time
from math import factorial

nums7 = [3, 11, 6, 2, 7, 13, 5]
t0 = time.perf_counter()
ans7 = largest_power_bruteforce(nums7)
dt = time.perf_counter() - t0
print(f"n = 7: {factorial(7)} hoán vị, chạy {dt:.3f} giây, đáp án {ans7}")

for n in [10, 12, 15]:
    print(f"n = {n}: n! * 15 * n ≈ {factorial(n) * 15 * n:.2e} bước")
import math
digits = math.floor(math.lgamma(50000 + 1) / math.log(10)) + 1   # log10(n!) = lgamma(n+1) / ln(10)
print("Số chữ số của (5*10^4)! ≈", f"{digits:,}")

n = 7: 5040 hoán vị, chạy 0.008 giây, đáp án [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 0, 5]
n = 10: n! * 15 * n ≈ 5.44e+08 bước
n = 12: n! * 15 * n ≈ 8.62e+10 bước
n = 15: n! * 15 * n ≈ 2.94e+14 bước
Số chữ số của (5*10^4)! ≈ 213,237


# 5. Bước đột phá 1 — "Tập hợp, không phải thứ tự"

Brute force phân biệt `n!` hoán vị. Nhưng thật ra `power` **không cần biết toàn bộ thứ tự**. Hãy xây dần 3 quan sát.

## Quan sát 1: `power` chỉ phụ thuộc dãy prefix AND

Theo mục 3.3: `power[i]` = số `k ∈ [1, n]` sao cho bit `14 - i` bật trong `A_k = perm[0] & ... & perm[k-1]`.

Lý do: dãy `A_1 ⊇ A_2 ⊇ ...` không tăng, nên các `k` có bit `b` bật trong `A_k` luôn là một **đoạn đầu** `1, 2, ..., P`. Đếm chúng ra đúng `P` = độ dài tiền tố mà mọi phần tử có bit `b`.

Cell dưới kiểm tra quan sát này trên nhiều hoán vị ngẫu nhiên.

In [14]:
import random

def power_via_prefix_and(perm) -> List[int]:
    power = [0] * BITS
    acc = (1 << BITS) - 1
    for x in perm:
        acc &= x                          # acc = A_k
        for i in range(BITS):
            if (acc >> (14 - i)) & 1:     # bit 14 - i còn sống trong A_k -> đếm thêm 1
                power[i] += 1
    return power

rng = random.Random(1)
ok = 0
for _ in range(500):
    perm = [rng.randrange(1 << 15) for _ in range(rng.randint(1, 8))]
    assert power_via_prefix_and(perm) == power_of(perm)
    ok += 1
print("Hai cách tính power trùng nhau trên", ok, "hoán vị ngẫu nhiên")

Hai cách tính power trùng nhau trên 500 hoán vị ngẫu nhiên


## Quan sát 2: `A_k` chỉ phụ thuộc **tập** `k` phần tử đầu

AND có tính giao hoán và kết hợp: `a & b & c = c & a & b`. Vậy `A_k` chỉ phụ thuộc **tập** `S_k` = {các phần tử đứng ở `k` vị trí đầu}, không phụ thuộc thứ tự bên trong.

Một hoán vị tương đương với một **chuỗi tập lồng nhau**: `∅ ⊂ S_1 ⊂ S_2 ⊂ ... ⊂ S_n = toàn bộ`, mỗi bước thêm đúng một phần tử.

## Quan sát 3: biến mục tiêu lexicographic thành một **tổng**

Dùng mẹo mục 3.7 với cơ số `n + 1` (vì mỗi `power[i] ≤ n`). Gán trọng số `(n+1)^b` cho bit `b` (bit 14 nặng nhất — đúng thứ tự ưu tiên). Định nghĩa:

```text
F(mask) = tổng (n+1)^b  với mọi bit b bật trong mask
```

Khi đó giá trị mã hoá của `power` là:

```text
encode(power) = Σ_b power_b · (n+1)^b = Σ_{k=1..n} F(A_k)
```

(mỗi `k` đóng góp `(n+1)^b` cho mọi bit `b` còn sống trong `A_k`). **Tối đa hoá `power` theo từ điển = tối đa hoá tổng `Σ F(A_k)`** — một bài toán "đường đi có tổng lớn nhất" qua các tập con.

In [15]:
from itertools import permutations

def F(mask, n):
    return sum((n + 1) ** b for b in range(BITS) if (mask >> b) & 1)

# Quan sát 2: cùng một tập {11, 3, 7}, mọi thứ tự cho cùng một AND
for p in permutations([11, 3, 7]):
    print(p, "-> AND =", p[0] & p[1] & p[2])

# Quan sát 3: encode(power) == tổng F(A_k)
perm = [9, 3, 6]
n = len(perm)
acc, total = (1 << BITS) - 1, 0
for x in perm:
    acc &= x
    total += F(acc, n)
power = power_of(perm)
enc = sum(power[14 - b] * (n + 1) ** b for b in range(BITS))
print("power =", power)
print("tổng F(A_k) =", total, "| encode(power) =", enc)

(11, 3, 7) -> AND = 3
(11, 7, 3) -> AND = 3
(3, 11, 7) -> AND = 3
(3, 7, 11) -> AND = 3
(7, 11, 3) -> AND = 3
(7, 3, 11) -> AND = 3
power = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2]
tổng F(A_k) = 66 | encode(power) = 66


# 6. Cách 2 — DP bitmask trên tập con

## Suy ra công thức truy hồi

Gọi `dp[S]` = giá trị lớn nhất của `Σ F(A_k)` khi **các phần tử của `S` chiếm `|S|` vị trí đầu** (theo một thứ tự tốt nhất), chỉ tính `k = 1..|S|`.

Xét phần tử **đứng cuối** trong `|S|` vị trí đầu, gọi là `x ∈ S`. Trước nó là một cách xếp của `S \ {x}`; prefix cuối cùng có AND = `AND(S)` — không phụ thuộc ai đứng cuối. Vậy:

```text
dp[∅] = 0
dp[S] = F(AND(S)) + max_{x ∈ S} dp[S \ {x}]
```

- `AND(S)` tính dần: `andv[S] = andv[S bỏ bit thấp nhất] & nums[chỉ số bit thấp nhất]`.
- Duyệt `S` tăng dần (mục 3.6) thì `S \ {x} < S` luôn có sẵn.
- Đáp án là `dp[full]`; giải mã cơ số `n + 1` để lấy lại `power`: `power[14 - b]` = chữ số thứ `b`.

Đây là ví dụ đẹp của việc **đổi "thứ tự" thành "tập"**: từ `n!` trạng thái xuống `2^n`.

In [16]:
def largest_power_dp(nums: List[int]) -> List[int]:
    n = len(nums)
    base = n + 1
    weight = [base ** b for b in range(BITS)]          # trọng số của bit b

    def F(mask):
        return sum(weight[b] for b in range(BITS) if (mask >> b) & 1)

    size = 1 << n
    andv = [0] * size
    andv[0] = (1 << BITS) - 1                          # AND của tập rỗng
    dp = [0] * size
    for S in range(1, size):
        low = S & -S
        idx = low.bit_length() - 1
        andv[S] = andv[S ^ low] & nums[idx]
        best_prev = max(dp[S ^ (1 << x)] for x in range(n) if (S >> x) & 1)
        dp[S] = F(andv[S]) + best_prev

    total = dp[size - 1]
    power = [0] * BITS
    for b in range(BITS):
        power[14 - b] = (total // weight[b]) % base    # chữ số thứ b ở cơ số n + 1
    return power


print(largest_power_dp([7, 5]))
print(largest_power_dp([3, 1, 7]))
print(largest_power_dp([9, 6, 3]))
print(largest_power_dp([3, 11, 6, 2, 7]))

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 2]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 5, 3]


## Nhìn bên trong bảng DP với `nums = [9, 6, 3]`

`n = 3` → cơ số 4. `F(9) = 4^3 + 4^0 = 65` (9 = 1001 có bit 3 và bit 0), `F(6) = 4^2 + 4^1 = 20`, `F(3) = 4 + 1 = 5`, `F(1) = 1`, `F(2) = 4`, `F(0) = 0`.

| Tập `S` | `AND(S)` | `F` | `dp[S]` | 4 chữ số cuối (bit 3,2,1,0) |
|---|---:|---:|---:|---|
| {9} | 9 | 65 | 65 | 1,0,0,1 |
| {6} | 6 | 20 | 20 | 0,1,1,0 |
| {9,6} | 0 | 0 | max(20, 65) + 0 = 65 | 1,0,0,1 |
| {3} | 3 | 5 | 5 | 0,0,1,1 |
| {9,3} | 1 | 1 | max(5, 65) + 1 = 66 | 1,0,0,2 |
| {6,3} | 2 | 4 | max(5, 20) + 4 = 24 | 0,1,2,0 |
| {9,6,3} | 0 | 0 | max(24, 66, 65) + 0 = 66 | **1,0,0,2** |

Đáp án `66` → giải mã ra `1,0,0,2` ở 4 ô cuối, khớp với bảng làm tay ở mục 2.2. Hàm `trace_dp` dưới in đúng bảng này.

In [17]:
def trace_dp(nums):
    n = len(nums)
    base = n + 1
    def F(mask):
        return sum(base ** b for b in range(BITS) if (mask >> b) & 1)
    andv = {0: (1 << BITS) - 1}
    dp = {0: 0}
    for S in range(1, 1 << n):
        low = S & -S
        andv[S] = andv[S ^ low] & nums[low.bit_length() - 1]
        dp[S] = F(andv[S]) + max(dp[S ^ (1 << x)] for x in range(n) if (S >> x) & 1)
        members = [nums[x] for x in range(n) if (S >> x) & 1]
        digits = [(dp[S] // base ** b) % base for b in (3, 2, 1, 0)]
        print(f"S={members!s:<10} AND={andv[S]:>2}  F={F(andv[S]):>3}  dp={dp[S]:>3}  bit3..0={digits}")

trace_dp([9, 6, 3])

S=[9]        AND= 9  F= 65  dp= 65  bit3..0=[1, 0, 0, 1]
S=[6]        AND= 6  F= 20  dp= 20  bit3..0=[0, 1, 1, 0]
S=[9, 6]     AND= 0  F=  0  dp= 65  bit3..0=[1, 0, 0, 1]
S=[3]        AND= 3  F=  5  dp=  5  bit3..0=[0, 0, 1, 1]
S=[9, 3]     AND= 1  F=  1  dp= 66  bit3..0=[1, 0, 0, 2]
S=[6, 3]     AND= 2  F=  4  dp= 24  bit3..0=[0, 1, 2, 0]
S=[9, 6, 3]  AND= 0  F=  0  dp= 66  bit3..0=[1, 0, 0, 2]


## Độ phức tạp của Cách 2 — tốt hơn nhiều nhưng vẫn chưa đủ

- Có `2^n` tập con; mỗi tập duyệt `n` phần tử để lấy `max` → **Time `O(2^n · n)`** (cộng `O(15)` cho `F`).
- Mảng `dp`, `andv` có `2^n` ô → **Space `O(2^n)`**.

| `n` | `n!` (Cách 1) | `2^n · n` (Cách 2) |
|---:|---:|---:|
| 10 | 3,628,800 | 10,240 |
| 20 | 2.4 · 10^18 | 2.1 · 10^7 |
| 50,000 | không tưởng | không tưởng |

Cách 2 dạy ta một ý tưởng quan trọng (thứ tự → tập), nhưng vẫn **mũ theo `n`**. Để xử lý `n = 5 · 10^4`, ta cần khai thác cấu trúc lexicographic **theo từng bit**.

# 7. Bước đột phá 2 — Greedy theo bit, từ cao xuống thấp

## 7.1. Lexicographic = tối ưu từng ô một, theo thứ tự ưu tiên

So sánh từ điển nói: **`power[0]` quan trọng tuyệt đối**. Vậy chiến lược tự nhiên là "lọc dần":

1. Trong mọi hoán vị, chỉ giữ những hoán vị có `power[0]` **lớn nhất**.
2. Trong số còn lại, chỉ giữ những hoán vị có `power[1]` lớn nhất.
3. ... lặp đến `power[14]`.

Cách lọc này **chắc chắn** cho đáp án đúng (đó chính là định nghĩa của "lớn nhất theo từ điển"). Câu hỏi là: **sau mỗi bước lọc, tập hoán vị "sống sót" trông như thế nào?** Nếu mô tả được nó gọn gàng, ta không cần liệt kê `n!` hoán vị nữa.

Thí nghiệm với `nums = [3, 11, 6, 2, 7]` (`11 = 1011`, `3 = 0011`, `7 = 0111`, `6 = 0110`, `2 = 0010`). Các bit 14..4 không ai có, nên mọi hoán vị đều được 0 — không lọc bớt ai. Ta xem các bit 3, 2, 1, 0.

In [18]:
from itertools import permutations

nums = [3, 11, 6, 2, 7]
alive = list(permutations(nums))
print("ban đầu:", len(alive), "hoán vị")
for i in range(11, 15):
    b = 14 - i
    best = max(power_of(p)[i] for p in alive)
    alive = [p for p in alive if power_of(p)[i] == best]
    firsts = sorted(set(p[0] for p in alive))
    print(f"bit {b}: power[{i}] tốt nhất = {best} -> còn {len(alive):>3} hoán vị; phần tử đầu có thể là {firsts}")
print("Các hoán vị sống sót cuối cùng:")
for p in alive:
    print("  ", list(p))

ban đầu: 120 hoán vị
bit 3: power[11] tốt nhất = 1 -> còn  24 hoán vị; phần tử đầu có thể là [11]
bit 2: power[12] tốt nhất = 0 -> còn  24 hoán vị; phần tử đầu có thể là [11]
bit 1: power[13] tốt nhất = 5 -> còn  24 hoán vị; phần tử đầu có thể là [11]
bit 0: power[14] tốt nhất = 3 -> còn   4 hoán vị; phần tử đầu có thể là [11]
Các hoán vị sống sót cuối cùng:
   [11, 3, 7, 6, 2]
   [11, 3, 7, 2, 6]
   [11, 7, 3, 6, 2]
   [11, 7, 3, 2, 6]


## 7.2. Đọc kết quả thí nghiệm → khái niệm "nhóm có thứ tự"

- **Bit 3:** chỉ `11` có bit 3. Muốn cột bit 3 dài nhất (= 1), `11` phải đứng đầu. Sống sót: 24 hoán vị = "`11` trước, 4 số còn lại **thứ tự tuỳ ý**". Ta mô tả tập này bằng **dãy nhóm**: `[[11], [3, 6, 2, 7]]` — các nhóm đứng theo thứ tự cố định, **bên trong mỗi nhóm thứ tự tự do**.
- **Bit 2:** nhóm đầu là `[11]`, mà `11` không có bit 2 → cột bit 2 gãy ngay hàng đầu, tốt nhất chỉ là 0, **không lọc thêm** (vẫn 24).
- **Bit 1:** `11` có bit 1; nhóm `[3,6,2,7]` cũng **toàn bộ** có bit 1 → cột dài trọn 5, vẫn 24 hoán vị.
- **Bit 0:** nhóm `[11]` có bit 0 → đi qua; nhóm `[3,6,2,7]` chỉ có `3, 7` mang bit 0 → phải xếp `3, 7` lên trước `6, 2` trong nhóm này. Dãy nhóm mới: `[[11], [3, 7], [6, 2]]` → `1 · 2! · 2! = 4` hoán vị, đúng như thí nghiệm.

## 7.3. Quy tắc tổng quát cho một bit `b`

Giả sử tập hoán vị sống sót đang là "mọi cách xếp tôn trọng dãy nhóm `G_0, G_1, ...`". Đi lần lượt qua các nhóm từ trái sang:

1. Nhóm nào **toàn bộ** phần tử có bit `b` → cả nhóm nằm trọn trong cột "chữ 1", cộng `len` vào `prefix` và đi tiếp.
2. Gặp nhóm đầu tiên `G_t` **không** toàn bộ có bit `b` (gọi là **nhóm biên**, *frontier*): chia nó thành `has` (có bit `b`) và `no` (không có).
   - Cột bit `b` dài nhất = `prefix + len(has)`: xếp `has` lên trước, phần tử đầu tiên của `no` làm gãy cột.
   - Nếu `has` khác rỗng: **tách** `G_t` thành `[has, no]` (thứ tự bắt buộc mới).
   - Nếu `has` rỗng: không tách gì, cột gãy ngay ở đầu `G_t`.
3. Nếu **mọi** nhóm đều toàn bộ có bit `b`: cột dài trọn `n`, không đổi gì.

Vì sao không làm hỏng các bit cao đã xét? Vì ta chỉ **sắp xếp lại bên trong một nhóm** — các ranh giới nhóm cũ (nơi những cột bit cao bị gãy) vẫn giữ nguyên.

In [19]:
def refine_one_bit(groups, b):
    """Áp dụng quy tắc 7.3 cho bit b. Trả về (P_b, dãy nhóm mới). Không sửa groups gốc."""
    groups = [list(g) for g in groups]
    prefix, t = 0, 0
    while t < len(groups) and all((x >> b) & 1 for x in groups[t]):
        prefix += len(groups[t])
        t += 1
    if t == len(groups):                       # mọi nhóm đều có bit b
        return prefix, groups
    has = [x for x in groups[t] if (x >> b) & 1]
    no = [x for x in groups[t] if not (x >> b) & 1]
    if has:
        groups[t:t+1] = [has, no]
    return prefix + len(has), groups

g = [[11], [3, 6, 2, 7]]
print("bit 2:", refine_one_bit(g, 2))   # nhóm biên [11] không có bit 2 -> P = 0, không tách
print("bit 1:", refine_one_bit(g, 1))   # mọi nhóm có bit 1 -> P = 5
print("bit 0:", refine_one_bit(g, 0))   # đi qua [11], tách [3,6,2,7] -> P = 1 + 2 = 3

bit 2: (0, [[11], [3, 6, 2, 7]])
bit 1: (5, [[11], [3, 6, 2, 7]])
bit 0: (3, [[11], [3, 7], [6, 2]])


## 7.4. Hai ý tưởng "nghe hợp lý" nhưng SAI

Trước khi viết lời giải, hãy loại bỏ hai lối tắt hấp dẫn:

**Sai lầm A — "đáp án chỉ là đếm":** `power[i] = số phần tử có bit 14 - i`. Đây chỉ là **cận trên** (một cột không thể dài hơn số số có bit đó). Nó sai vì các bit cao đã "chiếm" những vị trí đầu: ở `[9, 6, 3]`, bit 2 có 1 số (`6`) nhưng `9` buộc phải đứng đầu, nên cột bit 2 bằng 0.

**Sai lầm B — "sắp xếp giảm dần là xong":** số lớn hơn có bit cao hơn, nghe có vẻ đúng. Nhưng sau khi bit cao đã chốt, việc so "lớn hơn" vẫn ưu tiên các bit **đã chết** (vô nghĩa) thay vì các bit **còn sống**. Ở `[9, 6, 3]`: giảm dần cho `[9, 6, 3]` → bit 0 chỉ được 1, trong khi `[9, 3, 6]` được 2.

In [20]:
def count_only_wrong(nums):
    return [sum((x >> (14 - i)) & 1 for x in nums) for i in range(BITS)]

def sort_desc_wrong(nums):
    return power_of(sorted(nums, reverse=True))

for nums in ([9, 6, 3], [3, 11, 6, 2, 7]):
    print("nums =", nums)
    print("   đúng (brute force) :", largest_power_bruteforce(nums)[11:])
    print("   chỉ đếm (sai A)    :", count_only_wrong(nums)[11:])
    print("   giảm dần (sai B)   :", sort_desc_wrong(nums)[11:])

nums = [9, 6, 3]
   đúng (brute force) : [1, 0, 0, 2]
   chỉ đếm (sai A)    : [1, 1, 2, 2]
   giảm dần (sai B)   : [1, 0, 0, 1]
nums = [3, 11, 6, 2, 7]
   đúng (brute force) : [1, 0, 5, 3]
   chỉ đếm (sai A)    : [1, 2, 5, 3]
   giảm dần (sai B)   : [1, 0, 5, 2]


# 8. Cách 3 — Greedy theo bit + chia nhóm có thứ tự

## Thuật toán

```text
groups = [nums]                       # ban đầu: một nhóm chứa tất cả
for i in 0..14:
    b = 14 - i                        # bit cao xét trước
    prefix = 0, t = 0
    while t < số nhóm và groups[t] toàn bộ có bit b:
        prefix += len(groups[t]);  t += 1
    nếu t == số nhóm:  power[i] = n;  continue
    has, no = tách groups[t] theo bit b
    power[i] = prefix + len(has)
    nếu has khác rỗng: groups[t] -> [has, no]
```

Ghép các nhóm theo thứ tự ta cũng thu được **một hoán vị tối ưu cụ thể** (`perm`) — dùng để tự kiểm tra: `power_of(perm)` phải bằng `power` mà thuật toán trả về.

In [21]:
def largest_power_groups(nums: List[int], return_perm: bool = False):
    n = len(nums)
    groups = [list(nums)]
    power = [0] * BITS
    for i in range(BITS):
        b = 14 - i
        prefix, t = 0, 0
        while t < len(groups) and all((x >> b) & 1 for x in groups[t]):
            prefix += len(groups[t])
            t += 1
        if t == len(groups):                     # mọi phần tử đều có bit b
            power[i] = n
            continue
        has = [x for x in groups[t] if (x >> b) & 1]
        no = [x for x in groups[t] if not (x >> b) & 1]
        power[i] = prefix + len(has)
        if has:                                  # chỉ tách khi có cả hai phần
            groups[t:t+1] = [has, no]
    if return_perm:
        perm = [x for g in groups for x in g]
        return power, perm
    return power


for nums in ([7, 5], [3, 1, 7], [9, 6, 3], [3, 11, 6, 2, 7]):
    power, perm = largest_power_groups(nums, return_perm=True)
    assert power_of(perm) == power               # perm thật sự đạt được power này
    print(f"nums={nums!s:<17} perm={perm!s:<17} power={power}")

nums=[7, 5]            perm=[7, 5]            power=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 2]
nums=[3, 1, 7]         perm=[7, 3, 1]         power=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3]
nums=[9, 6, 3]         perm=[9, 3, 6]         power=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2]
nums=[3, 11, 6, 2, 7]  perm=[11, 3, 7, 6, 2]  power=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 5, 3]


## Độ phức tạp của Cách 3

Gọi `n = len(nums)`, `B = 15` (số bit).

- Mỗi bit: vòng `while` duyệt các nhóm đứng trước nhóm biên, `all(...)` kiểm tra từng phần tử → tối đa `O(n)`; tách nhóm biên `O(|G_t|) ≤ O(n)`.
- Tổng: **Time `O(B · n)` = `O(15 · n)`** ≈ `7.5 · 10^5` bước với `n = 5 · 10^4`.
- **Space `O(n)`** cho các nhóm.
- Mỗi bit tạo thêm **tối đa 1 nhóm**, nên luôn có **không quá 16 nhóm**. Ý này sẽ được khai thác ở Cách 4.

# 9. Mô phỏng (trace) Cách 3 với `nums = [3, 11, 6, 2, 7]`

Nhị phân 4 bit cuối: `3 = 0011`, `11 = 1011`, `6 = 0110`, `2 = 0010`, `7 = 0111`.

Ý nghĩa các cột: **nhóm trước** = dãy nhóm khi bắt đầu xét bit; **t** = chỉ số nhóm biên; **prefix** = tổng kích thước các nhóm đứng trước nhóm biên (đều có bit `b`); **has / no** = nhóm biên chia theo bit `b`; **P** = `power[14 - b]`.

| bit `b` | nhóm trước | t | prefix | has | no | P | nhóm sau |
|---|---|:---:|:---:|---|---|:---:|---|
| 14 … 4 | `[[3,11,6,2,7]]` | 0 | 0 | `[]` | cả nhóm | 0 | không đổi |
| 3 | `[[3,11,6,2,7]]` | 0 | 0 | `[11]` | `[3,6,2,7]` | 1 | `[[11],[3,6,2,7]]` |
| 2 | `[[11],[3,6,2,7]]` | 0 | 0 | `[]` | `[11]` | 0 | không đổi |
| 1 | `[[11],[3,6,2,7]]` | — | 5 | — | — | **5 = n** | không đổi (mọi nhóm đều có bit 1) |
| 0 | `[[11],[3,6,2,7]]` | 1 | 1 | `[3,7]` | `[6,2]` | 3 | `[[11],[3,7],[6,2]]` |

Kết quả: `power = [0,0,0,0,0,0,0,0,0,0,0,1,0,5,3]`, một hoán vị tối ưu: `perm = [11, 3, 7, 6, 2]`.

Bảng này có đủ **4 nhánh** của thuật toán: nhóm biên không ai có bit (bit 14..4 và bit 2), tách nhóm đầu (bit 3), mọi nhóm đều có bit (bit 1), và đi qua một nhóm "toàn có bit" rồi mới tách (bit 0).

In [22]:
def trace_groups(nums):
    n = len(nums)
    groups = [list(nums)]
    power = [0] * BITS
    print(f"nums = {nums}, n = {n}")
    for i in range(BITS):
        b = 14 - i
        before = [list(g) for g in groups]
        prefix, t = 0, 0
        while t < len(groups) and all((x >> b) & 1 for x in groups[t]):
            prefix += len(groups[t])
            t += 1
        if t == len(groups):
            power[i] = n
            print(f"bit {b:>2}: mọi nhóm đều có bit -> P = n = {n} | nhóm giữ nguyên {groups}")
            continue
        has = [x for x in groups[t] if (x >> b) & 1]
        no = [x for x in groups[t] if not (x >> b) & 1]
        power[i] = prefix + len(has)
        if has:
            groups[t:t+1] = [has, no]
            action = f"tách nhóm {t}: has={has}, no={no} -> {groups}"
        else:
            action = "nhóm biên không ai có bit -> không tách"
        print(f"bit {b:>2}: nhóm={before} t={t} prefix={prefix} P={power[i]} | {action}")
    perm = [x for g in groups for x in g]
    print("power =", power)
    print("perm  =", perm)
    return power

trace_groups([3, 11, 6, 2, 7]);

nums = [3, 11, 6, 2, 7], n = 5
bit 14: nhóm=[[3, 11, 6, 2, 7]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit 13: nhóm=[[3, 11, 6, 2, 7]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit 12: nhóm=[[3, 11, 6, 2, 7]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit 11: nhóm=[[3, 11, 6, 2, 7]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit 10: nhóm=[[3, 11, 6, 2, 7]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit  9: nhóm=[[3, 11, 6, 2, 7]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit  8: nhóm=[[3, 11, 6, 2, 7]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit  7: nhóm=[[3, 11, 6, 2, 7]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit  6: nhóm=[[3, 11, 6, 2, 7]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit  5: nhóm=[[3, 11, 6, 2, 7]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit  4: nhóm=[[3, 11, 6, 2, 7]] t=0 prefix=0 P=0 | nhóm biê

## Kiểm tra lại bằng hình ảnh lưới bit

Vẽ lưới của `perm = [11, 3, 7, 6, 2]` (hoán vị do greedy tạo ra) và so với `[11, 7, 6, 3, 2]` (sắp xếp giảm dần):

- Với greedy: cột 14 (bit 0) có `11, 3, 7` là `1`, đến `6` thì `0` → **3**.
- Với giảm dần: cột 14 có `11, 7` rồi gặp `6` → chỉ **2**.

Hai lưới giống nhau ở cột 11, 12, 13 (`1, 0, 5`) và khác ở cột 14 → greedy thắng.

In [23]:
p_greedy = show_grid([11, 3, 7, 6, 2])
print()
p_desc = show_grid(sorted([3, 11, 6, 2, 7], reverse=True))
print()
print("greedy > giảm dần ?", p_greedy > p_desc)

   cột i: 012345678901234
      11 | 000000000001011
       3 | 000000000000011
       7 | 000000000000111
       6 | 000000000000110
       2 | 000000000000010
power = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 5, 3]

   cột i: 012345678901234
      11 | 000000000001011
       7 | 000000000000111
       6 | 000000000000110
       3 | 000000000000011
       2 | 000000000000010
power = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 5, 2]

greedy > giảm dần ? True


# 10. Cách 4 — Greedy nén: `Counter` + AND của mỗi nhóm

Cách 3 đã đạt `O(15 · n)`. Nhưng có hai chỗ làm lặp lại công việc:

1. **Kiểm tra "cả nhóm có bit `b`"** bằng `all(...)` phải duyệt mọi phần tử. Thay vào đó, lưu **AND của cả nhóm** (`and_mask`): cả nhóm có bit `b` ⇔ `and_mask` có bit `b`. Kiểm tra còn `O(1)` mỗi nhóm, và chỉ có ≤ 16 nhóm.
2. **Giá trị trùng nhau luôn đi cùng nhau** (chúng có cùng mọi bit nên không bao giờ bị tách khỏi nhau). Vậy mỗi nhóm chỉ cần lưu `{giá trị: số lần}` và `size`. Tách nhóm chỉ tốn `O(số giá trị khác nhau trong nhóm)`.

Mỗi nhóm lưu bộ ba `[and_mask, size, counts]`. Khi tách, tính lại `and_mask` và `size` cho hai nửa.

**Độ phức tạp:** `O(n)` để đếm + mỗi bit `O(16)` quét nhóm + `O(D)` tách → **Time `O(n + 15 · D)`**, **Space `O(D)`**, với `D` = số giá trị khác nhau (`D ≤ min(n, 2^15)`). Đây là cùng một ý tưởng greedy với Cách 3, chỉ là **cách lưu trữ thông minh hơn** — nhanh hơn rõ ở các input có nhiều phần tử nhưng ít giá trị khác nhau.

In [24]:
from collections import Counter

FULL = (1 << BITS) - 1

def largest_power_counter(nums: List[int]) -> List[int]:
    n = len(nums)
    counts = Counter(nums)
    and_all = FULL
    for v in counts:
        and_all &= v
    groups = [[and_all, n, counts]]               # [and_mask, size, {value: count}]
    power = [0] * BITS
    for i in range(BITS):
        bit = 1 << (14 - i)
        prefix, t = 0, 0
        while t < len(groups) and groups[t][0] & bit:   # O(1) mỗi nhóm
            prefix += groups[t][1]
            t += 1
        if t == len(groups):
            power[i] = n
            continue
        has, no = {}, {}
        and_has = and_no = FULL
        size_has = size_no = 0
        for v, c in groups[t][2].items():
            if v & bit:
                has[v] = c; and_has &= v; size_has += c
            else:
                no[v] = c; and_no &= v; size_no += c
        power[i] = prefix + size_has
        if has:
            groups[t:t+1] = [[and_has, size_has, has], [and_no, size_no, no]]
    return power


for nums in ([7, 5], [3, 1, 7], [9, 6, 3], [3, 11, 6, 2, 7], [5, 5, 5]):
    print(f"{nums!s:<17} -> {largest_power_counter(nums)}")

[7, 5]            -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 2]
[3, 1, 7]         -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3]
[9, 6, 3]         -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2]
[3, 11, 6, 2, 7]  -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 5, 3]
[5, 5, 5]         -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 3]


## Thử tốc độ ở kích thước tối đa `n = 5 · 10^4`

Ba kiểu input: ngẫu nhiên; mọi phần tử bằng `32767` (mọi nhóm "toàn có bit" — nhánh `P = n` ở mọi bit); và dạng "bậc thang" `32767 >> (k % 16)` (nhiều giá trị lặp lại, nhóm biên tiến dần). Cả hai cách đều chạy chỉ vài phần trăm giây. Với input ngẫu nhiên hai cách tương đương (thời gian cụ thể dao động theo máy); Cách 4 nhanh hơn rõ rệt khi số giá trị khác nhau `D` nhỏ (hai input sau), vì nó không phải duyệt lại từng phần tử.

In [25]:
import random, time

rng = random.Random(4059)
inputs = {
    "ngẫu nhiên": [rng.randrange(1 << 15) for _ in range(50000)],
    "toàn 32767": [32767] * 50000,
    "bậc thang":  [32767 >> (k % 16) for k in range(50000)],
}
for name, arr in inputs.items():
    t0 = time.perf_counter(); r3 = largest_power_groups(arr); t1 = time.perf_counter()
    r4 = largest_power_counter(arr); t2 = time.perf_counter()
    assert r3 == r4
    print(f"{name:<11}: Cách 3 {t1 - t0:.4f}s | Cách 4 {t2 - t1:.4f}s | power[:4] = {r3[:4]}")

ngẫu nhiên : Cách 3 0.0077s | Cách 4 0.0096s | power[:4] = [25096, 12704, 6364, 3212]


toàn 32767 : Cách 3 0.0237s | Cách 4 0.0011s | power[:4] = [50000, 50000, 50000, 50000]


bậc thang  : Cách 3 0.0287s | Cách 4 0.0016s | power[:4] = [3125, 6250, 9375, 12500]


# 11. Phiên bản nộp LeetCode

Code nộp bài nên **gọn, không `print`**, đúng signature. Ta dùng Cách 4 (nhanh, ít bộ nhớ). Nhớ:

- `from typing import List` khi chạy ngoài LeetCode (trên LeetCode đã có sẵn).
- Tên method `lexLargestPower` là **giả định** (mirror không cho skeleton) — đổi theo tên LeetCode hiển thị nếu khác.

Nếu bạn thích code ngắn và dễ nhớ hơn, Cách 3 cũng hoàn toàn đủ nhanh để nộp (chỉ `O(15 · n)`).

In [26]:
from typing import List
from collections import Counter

class Solution:
    def lexLargestPower(self, nums: List[int]) -> List[int]:
        FULL = (1 << 15) - 1
        n = len(nums)
        counts = Counter(nums)
        and_all = FULL
        for v in counts:
            and_all &= v
        groups = [[and_all, n, counts]]          # [AND của nhóm, kích thước, {giá trị: số lần}]
        power = [0] * 15
        for i in range(15):
            bit = 1 << (14 - i)
            prefix, t = 0, 0
            while t < len(groups) and groups[t][0] & bit:
                prefix += groups[t][1]
                t += 1
            if t == len(groups):
                power[i] = n
                continue
            has, no = {}, {}
            and_has = and_no = FULL
            size_has = size_no = 0
            for v, c in groups[t][2].items():
                if v & bit:
                    has[v] = c; and_has &= v; size_has += c
                else:
                    no[v] = c; and_no &= v; size_no += c
            power[i] = prefix + size_has
            if has:
                groups[t:t + 1] = [[and_has, size_has, has], [and_no, size_no, no]]
        return power

In [27]:
sol = Solution()

tests = [
    ([7, 5],            [0]*12 + [2, 1, 2]),     # ví dụ 1
    ([3, 1, 7],         [0]*12 + [1, 2, 3]),     # ví dụ 2
    ([0],               [0]*15),                 # n = 1, không bit nào
    ([32767],           [1]*15),                 # n = 1, mọi bit
    ([5, 5, 5],         [0]*12 + [3, 0, 3]),     # toàn giá trị trùng
    ([0, 32767],        [1]*15),                 # 32767 phải đứng đầu
    ([16384, 1],        [1] + [0]*14),           # bit 14 chốt 16384 đứng đầu
    ([9, 6, 3],         [0]*11 + [1, 0, 0, 2]),  # giảm dần sẽ sai
    ([3, 11, 6, 2, 7],  [0]*11 + [1, 0, 5, 3]),  # ví dụ trace
    ([32767] * 50000,   [50000]*15),             # n tối đa
]
for nums, expected in tests:
    result = sol.lexLargestPower(nums)
    assert result == expected, (nums[:5], result, expected)
    label = str(nums) if len(nums) <= 6 else f"[32767] * {len(nums)}"
    print(f"PASS  {label:<20} -> {result if len(nums) <= 6 else result[:3] + ['...']}")
print("Tất cả test PASS!")

PASS  [7, 5]               -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 2]
PASS  [3, 1, 7]            -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3]
PASS  [0]                  -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
PASS  [32767]              -> [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
PASS  [5, 5, 5]            -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 3]
PASS  [0, 32767]           -> [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
PASS  [16384, 1]           -> [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
PASS  [9, 6, 3]            -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2]
PASS  [3, 11, 6, 2, 7]     -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 5, 3]
PASS  [32767] * 50000      -> [50000, 50000, 50000, '...']
Tất cả test PASS!


# 12. Kiểm thử chéo (cross-check) mọi cách với brute force

Brute force đúng hiển nhiên theo đề nên là "trọng tài". Ta sinh nhiều input nhỏ ngẫu nhiên (seed cố định để chạy lại ra y hệt), gồm cả số nhỏ (nhiều bit trùng nhau, nhiều giá trị lặp) lẫn số đủ 15 bit, rồi so:

- Cách 1 (brute force) = Cách 2 (DP) = Cách 3 (nhóm) = Cách 4 (Counter) = `Solution`;
- hoán vị `perm` do Cách 3 tạo ra là hoán vị của `nums` và `power_of(perm)` đúng bằng đáp án.

(Ngoài notebook, mình đã chạy 3000 ca `n ≤ 7` + 300 ca `n = 8..12` (DP so với greedy) — tất cả khớp.)

In [28]:
import random

rng = random.Random(2024)
cases = 0
for _ in range(250):
    n = rng.randint(1, 6)
    kind = rng.random()
    if kind < 0.5:
        nums = [rng.randrange(16) for _ in range(n)]               # 4 bit: nhiều trùng lặp
    elif kind < 0.8:
        nums = [rng.randrange(1 << 15) for _ in range(n)]          # đủ 15 bit
    else:
        nums = [rng.choice([0, 1, 3, 5, 6, 7, 2, 4]) for _ in range(n)]
    expected = largest_power_bruteforce(nums)
    power3, perm3 = largest_power_groups(nums, return_perm=True)
    assert largest_power_dp(nums) == expected, nums
    assert power3 == expected, nums
    assert largest_power_counter(nums) == expected, nums
    assert sol.lexLargestPower(nums) == expected, nums
    assert sorted(perm3) == sorted(nums) and power_of(perm3) == expected, nums
    cases += 1
print(f"Cross-check: {cases} ca ngẫu nhiên, mọi cách đều khớp brute force.")

Cross-check: 250 ca ngẫu nhiên, mọi cách đều khớp brute force.


# 13. Vì sao greedy đúng? (chứng minh ngắn bằng invariant)

Gọi một hoán vị là **tôn trọng** dãy nhóm `G_0, G_1, ..., G_m` nếu nó gồm các phần tử của `G_0` (thứ tự tuỳ ý), rồi các phần tử của `G_1` (thứ tự tuỳ ý), ...

**Invariant.** Sau khi xử lý các bit `14, 13, ..., b+1`:

> Một hoán vị đạt giá trị **tốt nhất có thể** cho `power[0..13-b]` (các ô ứng với bit đã xét) **khi và chỉ khi** nó tôn trọng dãy nhóm hiện tại.

**Khởi đầu:** chưa xét bit nào, một nhóm duy nhất chứa tất cả → mọi hoán vị đều tôn trọng. Đúng.

**Bước chuyển (xét bit `b`).** Chỉ cần xét các hoán vị tôn trọng dãy nhóm (theo invariant, những hoán vị khác đã thua ở một ô trước đó).

- Các nhóm `G_0..G_{t-1}` **toàn bộ** có bit `b`: dù xếp trong nhóm thế nào, chúng đều nằm trong cột "chữ 1" → đóng góp đúng `prefix`.
- Nhóm biên `G_t` có ít nhất một phần tử **không** có bit `b`, nên cột không thể vượt qua hết `G_t`. Cột dài nhất có thể là `prefix + |has|`, đạt được **khi và chỉ khi** mọi phần tử của `has` đứng trước mọi phần tử của `no` bên trong `G_t`. Các nhóm sau `G_t` không ảnh hưởng (cột đã gãy).
- Vậy tập hoán vị tối ưu mới = tôn trọng dãy nhóm sau khi tách `G_t → [has, no]`. Invariant tiếp tục đúng. (Nếu mọi nhóm đều toàn có bit `b`, cột dài `n` với mọi hoán vị tôn trọng, không cần đổi.)

**Không phá kết quả cũ:** tách chỉ thêm ràng buộc **bên trong** một nhóm, nên hoán vị tôn trọng dãy mới cũng tôn trọng dãy cũ → các ô `power` đã chốt vẫn đạt tối ưu.

Sau bit 0, mọi ô đều tối ưu theo đúng thứ tự ưu tiên → đó chính là vector lớn nhất theo từ điển. ∎

**Vì sao Cách 2 (DP) đúng:** mục tiêu lexicographic được mã hoá thành tổng `Σ F(A_k)` (mỗi chữ số ≤ `n` < cơ số, nên không có "nhớ" giữa các chữ số), và `A_k` chỉ phụ thuộc tập `S_k` → bài toán có cấu trúc con tối ưu trên các tập con.

# 14. Độ phức tạp và so sánh các cách

Ký hiệu: `n = len(nums)`, `B = 15` (số bit), `D` = số giá trị khác nhau (`D ≤ min(n, 2^15)`).

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Dùng khi |
|---|---|---|---|---|
| 1. Brute force | thử mọi hoán vị | `O(n! · B · n)` | `O(n)` | `n ≤ 8`, làm trọng tài kiểm thử |
| 2. DP bitmask | tập thay cho thứ tự, mã hoá cơ số `n+1` | `O(2^n · n)` | `O(2^n)` | `n ≤ ~18`, học ý tưởng |
| 3. Greedy + nhóm | lọc lexicographic theo bit, tách nhóm biên | `O(B · n)` | `O(n)` | nộp bài, dễ nhớ |
| 4. Greedy nén | như 3 + `Counter` + AND của nhóm | `O(n + B · D)` | `O(D)` | nộp bài, nhanh nhất |

Ghi chú:

- `B = 15` là hằng số nên Cách 3 thực chất là **tuyến tính**: với `n = 5 · 10^4`, chỉ vài phần trăm giây trong Python (xem cell đo thời gian ở mục 10).
- Số nhóm luôn ≤ `B + 1 = 16` vì mỗi bit tách tối đa một nhóm.
- Không cần sắp xếp (`sort`), nên không có thừa số `log n`.

# 15. Những lỗi thường gặp

## Lỗi 1 — Đảo nhầm chỉ số: dùng bit `i` cho `power[i]`
Đề nói `power[i]` ứng với bit `14 - i`. Dùng bit `i` sẽ ra vector **bị lật ngược** (và còn tối ưu sai thứ tự ưu tiên!).
**Sửa:** luôn viết `b = 14 - i`.

## Lỗi 2 — Độ ưu tiên toán tử: `x & 1 << b == 1`
Python hiểu là `(x & (1 << b)) == 1`. Với `b ≥ 1`, `x & (1 << b)` bằng `0` hoặc `2^b` — **không bao giờ bằng 1**.
**Sửa:** `(x >> b) & 1 == 1` hoặc `x & (1 << b) != 0`, và thêm ngoặc cho chắc.

## Lỗi 3 — Chỉ đếm số phần tử có bit (bỏ qua ràng buộc từ bit cao)
Đó chỉ là cận trên. Sai ở `[9, 6, 3]` (mục 7.4).

## Lỗi 4 — Sắp xếp giảm dần
Sai ở `[9, 6, 3]`: giảm dần ưu tiên cả những bit đã "chết". (Mục 7.4.)

## Lỗi 5 — Chỉ nhìn nhóm đầu tiên
Nếu nhóm đầu **toàn bộ** có bit `b`, cột còn kéo dài sang nhóm sau. Code chỉ xét `groups[0]` sẽ sai ở `[3, 11, 6, 2, 7]` (bit 1 phải là 5, code sai cho 1).

## Lỗi 6 — Quên nhánh "mọi nhóm đều có bit"
Khi vòng `while` chạy hết các nhóm, `t == len(groups)`; nếu vẫn truy cập `groups[t]` → `IndexError`. Ví dụ `[7, 7]`: mọi bit 2, 1, 0 đều do cả hai số mang.
**Sửa:** kiểm tra `if t == len(groups): power[i] = n; continue`.

Cell dưới chạy thử từng lỗi.

In [29]:
def wrong_index(nums):                      # Lỗi 1: dùng bit i thay vì bit 14 - i
    groups, power = [list(nums)], [0] * 15
    for i in range(15):
        b = i
        p, groups = refine_one_bit(groups, b)
        power[i] = p
    return power

def has_bit_wrong(x, b):                    # Lỗi 2: sai độ ưu tiên
    return x & 1 << b == 1

def only_first_group_wrong(nums):           # Lỗi 5: chỉ xét groups[0]
    groups, power = [list(nums)], [0] * 15
    for i in range(15):
        b = 14 - i
        g = groups[0]
        has = [x for x in g if (x >> b) & 1]
        no = [x for x in g if not (x >> b) & 1]
        power[i] = len(has)
        if has and no:
            groups[0:1] = [has, no]
    return power

def forget_full_case_wrong(nums):           # Lỗi 6: quên nhánh t == len(groups)
    groups, power = [list(nums)], [0] * 15
    for i in range(15):
        b = 14 - i
        prefix, t = 0, 0
        while t < len(groups) and all((x >> b) & 1 for x in groups[t]):
            prefix += len(groups[t]); t += 1
        has = [x for x in groups[t] if (x >> b) & 1]   # IndexError khi t == len(groups)
        power[i] = prefix + len(has)
    return power

print("Lỗi 1:", wrong_index([7, 5]), "| đúng:", largest_power_groups([7, 5]))
print("Lỗi 2: has_bit_wrong(6, 1) =", has_bit_wrong(6, 1), "| đúng:", (6 >> 1) & 1 == 1)
print("Lỗi 5:", only_first_group_wrong([3, 11, 6, 2, 7])[11:], "| đúng:", largest_power_groups([3, 11, 6, 2, 7])[11:])
try:
    forget_full_case_wrong([7, 7])
except IndexError as e:
    print("Lỗi 6: IndexError ->", e)

Lỗi 1: [2, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] | đúng: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 2]
Lỗi 2: has_bit_wrong(6, 1) = False | đúng: True
Lỗi 5: [1, 0, 1, 1] | đúng: [1, 0, 5, 3]
Lỗi 6: IndexError -> list index out of range


## Edge cases cần nhớ

| Trường hợp | Ví dụ | Kết quả | Vì sao |
|---|---|---|---|
| `n = 1` | `[0]` | toàn `0` | không bit nào bật |
| `n = 1`, mọi bit | `[32767]` | toàn `1` | một phần tử có cả 15 bit |
| Toàn giá trị trùng | `[5, 5, 5]` | `..., 3, 0, 3` | nhóm không bao giờ bị tách; bit có/không như nhau cho cả nhóm |
| Có số 0 | `[0, 32767]` | toàn `1` | số 0 luôn bị đẩy ra sau ngay ở bit cao nhất có người mang |
| Bit 14 | `[16384, 1]` | `[1, 0, ..., 0]` | bit 14 thắng tuyệt đối dù `1` có bit 0 |
| Mọi phần tử chung một bit | `[32767] * 50000` | toàn `50000` | nhánh `P = n` |

Tất cả đã được `assert` trong mục 11.

Một lưu ý nhỏ: `power[i] = 0` **không** có nghĩa là không ai có bit `14 - i` — chỉ có nghĩa là phần tử **đứng đầu** (bị các bit cao hơn ép) không có bit đó.

# 16. Mẹo nhận dạng và bài liên quan

## Dấu hiệu nhận dạng

1. **"Lexicographically largest/smallest" + các "tiêu chí" có thứ tự ưu tiên rõ ràng** → nghĩ ngay tới **greedy lọc dần**: tối ưu tiêu chí 1, *giữ lại tập lời giải tối ưu*, rồi tối ưu tiêu chí 2 trong tập đó, ... Mấu chốt là tìm cách **mô tả gọn tập lời giải còn sống** (ở đây: dãy nhóm có thứ tự).
2. **"`j` phần tử đầu tiên đều có tính chất X"** → nghĩ tới **prefix AND** (hoặc prefix min/max): đại lượng đơn điệu theo tiền tố.
3. **Giá trị < `2^15`, `2^20`, `2^30`** → vòng lặp ngoài theo **bit**, tổng chi phí `O(số bit · n)`.
4. **"Được sắp xếp lại tuỳ ý"** → thứ tự bên trong một khối thường không quan trọng; hãy nghĩ theo **tập / nhóm / tần suất (`Counter`)** thay vì hoán vị.
5. **Mục tiêu lexicographic với các ô bị chặn** (`0 ≤ ô ≤ n`) → có thể **mã hoá thành một số** cơ số `n + 1` để cộng dồn / dùng DP.

## Bài liên quan

- **LeetCode 2275 — Largest Combination With Bitwise AND Greater Than Zero**: đáp án = số phần tử có bit `b` nhiều nhất (chính là "cận trên" `count_b` của bài này). Mini code bên dưới.
- **LeetCode 421 — Maximum XOR of Two Numbers in an Array**: greedy theo bit từ cao xuống thấp, bit cao quyết định trước.
- **LeetCode 2411 — Smallest Subarrays With Maximum Bitwise OR**: theo dõi từng bit trên đoạn con, đại lượng OR đơn điệu.
- **LeetCode 179 — Largest Number**: "sắp xếp lại để được kết quả lớn nhất", nhưng ở đó có phép so sánh cặp; ở bài này thì **không** có thứ tự sắp xếp toàn cục đơn giản (xem lỗi "giảm dần").
- **LeetCode 316 — Remove Duplicate Letters**: một bài greedy lexicographic kinh điển khác.

In [30]:
def largest_combination_and(candidates):
    """LeetCode 2275: tập con lớn nhất có AND > 0 = số phần tử có chung một bit nhiều nhất."""
    return max(sum((x >> b) & 1 for x in candidates) for b in range(24))

print(largest_combination_and([16, 17, 71, 62, 12, 24, 14]))   # 4
print(largest_combination_and([8, 8]))                         # 2

# Liên hệ với bài 4059: power[i] luôn <= số phần tử có bit 14 - i
nums = [3, 11, 6, 2, 7]
print(all(p <= c for p, c in zip(largest_power_counter(nums), count_only_wrong(nums))))

4
2
True


# 17. Bài tập vận dụng

Làm theo thứ tự, **tự làm trước** rồi mới xem gợi ý và đáp án ở cuối notebook.

## Bài 1 — Đoán output (không chạy code)

a) Với `perm = [6, 7, 5, 4]` (6 = 110, 7 = 111, 5 = 101, 4 = 100), ba ô cuối `power[12], power[13], power[14]` bằng bao nhiêu? (Đây là `power` của **hoán vị cố định này**, không phải đáp án tối ưu.)

b) `format(13, '015b')[11]` in ra ký tự gì, và ký tự đó là bit thứ mấy của 13?

## Bài 2 — Tính tay thuật toán nhóm

Với `nums = [5, 12, 7, 13, 4]` (`5 = 0101`, `12 = 1100`, `7 = 0111`, `13 = 1101`, `4 = 0100`), hãy lập bảng như mục 9 cho bit 3, 2, 1, 0:

| bit | nhóm trước | t | prefix | has | no | P | nhóm sau |
|---|---|---|---|---|---|---|---|

Cho biết 4 ô cuối của `power` và một hoán vị tối ưu.

## Bài 3 — Điền TODO: tính `power` của một hoán vị bằng prefix AND

Không dùng vòng `while` theo cột; hãy dùng biến `acc` (prefix AND) như Quan sát 1 ở mục 5.

In [31]:
def exercise_power_prefix_and(perm):
    power = [0] * 15
    acc = (1 << 15) - 1
    # TODO 1: duyệt từng x trong perm
    # TODO 2: cập nhật acc = AND của các phần tử đã duyệt
    # TODO 3: với mỗi i, nếu bit (14 - i) còn bật trong acc thì power[i] += 1
    raise NotImplementedError("Hãy tự hoàn thiện hàm này")

# Tự làm xong rồi mới mở test:
# assert exercise_power_prefix_and([7, 5]) == [0]*12 + [2, 1, 2]
# assert exercise_power_prefix_and([6, 7, 5, 4])[12:] == [4, 2, 0]

## Bài 4 — Điền TODO: một bước tách nhóm

Viết hàm nhận dãy nhóm `groups` và bit `b`, trả về `(P_b, groups_moi)` theo quy tắc 7.3. Không được sửa `groups` gốc.

In [32]:
def exercise_refine(groups, b):
    groups = [list(g) for g in groups]
    prefix, t = 0, 0
    # TODO 1: while t < len(groups) và mọi x trong groups[t] có bit b: cộng prefix, tăng t
    # TODO 2: nếu t == len(groups): trả về (prefix, groups)
    # TODO 3: tách groups[t] thành has / no theo bit b
    # TODO 4: nếu has khác rỗng: groups[t:t+1] = [has, no]
    # TODO 5: trả về (prefix + len(has), groups)
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_refine([[11], [3, 6, 2, 7]], 0) == (3, [[11], [3, 7], [6, 2]])
# assert exercise_refine([[11], [3, 6, 2, 7]], 1) == (5, [[11], [3, 6, 2, 7]])
# assert exercise_refine([[11], [3, 6, 2, 7]], 2) == (0, [[11], [3, 6, 2, 7]])

## Bài 5 — Tự viết lại + biến thể: thứ tự ưu tiên bit tuỳ ý

Biến thể: cho thêm danh sách `order` gồm 15 bit khác nhau; định nghĩa `power[i]` là độ dài tiền tố mà mọi phần tử có bit `order[i]`. Bài gốc ứng với `order = [14, 13, ..., 0]`.

Viết `exercise_power_with_order(nums, order)` trả về `power` lớn nhất theo từ điển. Gợi ý: greedy không hề dùng việc "bit 14 là bit cao nhất" — nó chỉ dùng **thứ tự ưu tiên**.

In [33]:
def exercise_power_with_order(nums, order):
    # TODO: viết lại greedy chia nhóm, nhưng ở bước i dùng b = order[i]
    raise NotImplementedError("Hãy tự viết lại greedy")

# Tự làm xong rồi mới mở test:
# low_first = list(range(15))                      # ưu tiên bit 0 trước
# assert exercise_power_with_order([9, 6, 3], list(range(14, -1, -1))) == [0]*11 + [1, 0, 0, 2]
# assert exercise_power_with_order([9, 6, 3], low_first)[:4] == [2, 1, 0, 0]

## Bài 6 — Mini challenge: có bao nhiêu hoán vị tối ưu?

Đếm số hoán vị **theo chỉ số** (coi các phần tử ở vị trí khác nhau là khác nhau, kể cả khi trùng giá trị) đạt đúng `power` lớn nhất.

Ví dụ: `[3, 11, 6, 2, 7]` có **4** hoán vị tối ưu (xem thí nghiệm mục 7.1); `[5, 5, 5]` có **6**; `[9, 6, 3]` có **1**.

Yêu cầu: chạy nhanh với `n` lớn (không liệt kê hoán vị). Gợi ý: nghĩ về invariant ở mục 13.

In [34]:
def exercise_count_optimal(nums):
    # TODO: chạy greedy chia nhóm tới hết, rồi suy ra số hoán vị từ kích thước các nhóm
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_count_optimal([3, 11, 6, 2, 7]) == 4
# assert exercise_count_optimal([5, 5, 5]) == 6
# assert exercise_count_optimal([9, 6, 3]) == 1

## Gợi ý (3 mức: hướng tư duy → cấu trúc/cú pháp → gần-code)

**Bài 1**
1. Vẽ lưới bit 3 cột cuối rồi đếm "chữ 1 liên tiếp từ trên xuống".
2. Cột bit 2: cả bốn số đều ≥ 4. Cột bit 1: `6, 7` có, `5` không. Cột bit 0: `6` chẵn.
3. Ký tự thứ `i` của chuỗi 15 ký tự là bit `14 - i`; `13 = 1101`.

**Bài 2**
1. Bit 3: ai có bit 3 (≥ 8)? Họ tạo nhóm đầu.
2. Bit 2: kiểm tra **mọi** nhóm — có thể tất cả đều có bit 2!
3. Bit 1: nhóm biên là nhóm đầu; có ai trong nhóm đầu mang bit 1 không? Bit 0: tách nhóm đầu theo tính chẵn lẻ.

**Bài 3**
1. `power[i]` = số `k` mà bit `14 - i` còn sống trong prefix AND `A_k`.
2. `for x in perm: acc &= x` rồi `for i in range(15): ...`.
3. `if (acc >> (14 - i)) & 1: power[i] += 1`.

**Bài 4**
1. Ba nhánh: đi qua nhóm "toàn có bit", gặp nhóm biên, hoặc đi hết.
2. `all((x >> b) & 1 for x in groups[t])`, list comprehension cho `has`/`no`, gán lát cắt `groups[t:t+1]`.
3. Chép đúng khung của `refine_one_bit` ở mục 7.3 — nhưng hãy tự gõ lại từng dòng.

**Bài 5**
1. Chứng minh ở mục 13 không dùng gì về "giá trị" của bit, chỉ dùng thứ tự ưu tiên.
2. Vòng `for i in range(15): b = order[i]`, phần còn lại giữ nguyên như Cách 3.
3. Để kiểm tra, viết brute force riêng: `power_custom(perm, order)` + `max` trên `permutations`.

**Bài 6**
1. Theo invariant, hoán vị tối ưu = hoán vị **tôn trọng dãy nhóm cuối cùng**.
2. Bên trong mỗi nhóm kích thước `s`, có `s!` cách xếp (theo chỉ số); các nhóm độc lập.
3. `math.prod(math.factorial(len(g)) for g in groups)`.

# Đáp án tham khảo

Chỉ xem sau khi đã tự làm. Mỗi đáp án có `assert` để tự kiểm chứng.

## Đáp án Bài 1

a) Bit 2: `6, 7, 5, 4` đều có → **4**. Bit 1: `6, 7` có, `5 = 101` không → **2**. Bit 0: `6 = 110` không có ngay hàng đầu → **0**. Vậy `[4, 2, 0]`.

b) `format(13, '015b') = '000000000001101'`, ký tự thứ 11 là `'1'`, chính là **bit 14 - 11 = 3** của 13 (`13 = 8 + 4 + 1`).

In [35]:
assert power_of([6, 7, 5, 4])[12:] == [4, 2, 0]
s = format(13, '015b')
assert s[11] == '1' and (13 >> 3) & 1 == 1
print(s, "->", s[11])
show_grid([6, 7, 5, 4]);

000000000001101 -> 1
   cột i: 012345678901234
       6 | 000000000000110
       7 | 000000000000111
       5 | 000000000000101
       4 | 000000000000100
power = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 2, 0]


## Đáp án Bài 2

| bit | nhóm trước | t | prefix | has | no | P | nhóm sau |
|---|---|:---:|:---:|---|---|:---:|---|
| 3 | `[[5,12,7,13,4]]` | 0 | 0 | `[12,13]` | `[5,7,4]` | 2 | `[[12,13],[5,7,4]]` |
| 2 | `[[12,13],[5,7,4]]` | — | 5 | — | — | **5 = n** | không đổi (mọi số đều có bit 2) |
| 1 | `[[12,13],[5,7,4]]` | 0 | 0 | `[]` | `[12,13]` | 0 | không đổi |
| 0 | `[[12,13],[5,7,4]]` | 0 | 0 | `[13]` | `[12]` | 1 | `[[13],[12],[5,7,4]]` |

4 ô cuối: `[2, 5, 0, 1]`; một hoán vị tối ưu: `[13, 12, 5, 7, 4]`.

In [36]:
assert trace_groups([5, 12, 7, 13, 4])[11:] == [2, 5, 0, 1]
assert largest_power_bruteforce([5, 12, 7, 13, 4])[11:] == [2, 5, 0, 1]

nums = [5, 12, 7, 13, 4], n = 5
bit 14: nhóm=[[5, 12, 7, 13, 4]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit 13: nhóm=[[5, 12, 7, 13, 4]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit 12: nhóm=[[5, 12, 7, 13, 4]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit 11: nhóm=[[5, 12, 7, 13, 4]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit 10: nhóm=[[5, 12, 7, 13, 4]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit  9: nhóm=[[5, 12, 7, 13, 4]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit  8: nhóm=[[5, 12, 7, 13, 4]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit  7: nhóm=[[5, 12, 7, 13, 4]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit  6: nhóm=[[5, 12, 7, 13, 4]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit  5: nhóm=[[5, 12, 7, 13, 4]] t=0 prefix=0 P=0 | nhóm biên không ai có bit -> không tách
bit  4: nhóm=[[5, 12, 7, 13, 4]] t=0 prefix=0 P=

## Đáp án Bài 3 và Bài 4

In [37]:
def answer_power_prefix_and(perm):
    power = [0] * 15
    acc = (1 << 15) - 1
    for x in perm:
        acc &= x
        for i in range(15):
            if (acc >> (14 - i)) & 1:
                power[i] += 1
    return power


def answer_refine(groups, b):
    groups = [list(g) for g in groups]
    prefix, t = 0, 0
    while t < len(groups) and all((x >> b) & 1 for x in groups[t]):
        prefix += len(groups[t])
        t += 1
    if t == len(groups):
        return prefix, groups
    has = [x for x in groups[t] if (x >> b) & 1]
    no = [x for x in groups[t] if not (x >> b) & 1]
    if has:
        groups[t:t+1] = [has, no]
    return prefix + len(has), groups


assert answer_power_prefix_and([7, 5]) == [0]*12 + [2, 1, 2]
assert answer_power_prefix_and([6, 7, 5, 4])[12:] == [4, 2, 0]
assert answer_refine([[11], [3, 6, 2, 7]], 0) == (3, [[11], [3, 7], [6, 2]])
assert answer_refine([[11], [3, 6, 2, 7]], 1) == (5, [[11], [3, 6, 2, 7]])
assert answer_refine([[11], [3, 6, 2, 7]], 2) == (0, [[11], [3, 6, 2, 7]])
print("Bài 3, Bài 4: PASS")

Bài 3, Bài 4: PASS


## Đáp án Bài 5 — greedy với thứ tự ưu tiên bit tuỳ ý

Kiểm chứng bằng brute force tự viết cho định nghĩa mới, với hai thứ tự: thứ tự gốc và "bit thấp trước".

In [38]:
from itertools import permutations
import random

def answer_power_with_order(nums, order):
    n = len(nums)
    groups = [list(nums)]
    power = [0] * len(order)
    for i, b in enumerate(order):
        prefix, t = 0, 0
        while t < len(groups) and all((x >> b) & 1 for x in groups[t]):
            prefix += len(groups[t]); t += 1
        if t == len(groups):
            power[i] = n
            continue
        has = [x for x in groups[t] if (x >> b) & 1]
        no = [x for x in groups[t] if not (x >> b) & 1]
        power[i] = prefix + len(has)
        if has:
            groups[t:t+1] = [has, no]
    return power

def power_custom(perm, order):
    res = []
    for b in order:
        j = 0
        while j < len(perm) and (perm[j] >> b) & 1:
            j += 1
        res.append(j)
    return res

high_first = list(range(14, -1, -1))
low_first = list(range(15))
assert answer_power_with_order([9, 6, 3], high_first) == [0]*11 + [1, 0, 0, 2]
assert answer_power_with_order([9, 6, 3], low_first)[:4] == [2, 1, 0, 0]

rng = random.Random(5)
for _ in range(150):
    nums = [rng.randrange(16) for _ in range(rng.randint(1, 5))]
    order = rng.sample(range(15), 15)
    best = max(power_custom(p, order) for p in permutations(nums))
    assert answer_power_with_order(nums, order) == best, (nums, order)
print("Bài 5: PASS (150 ca ngẫu nhiên với thứ tự bit ngẫu nhiên)")

Bài 5: PASS (150 ca ngẫu nhiên với thứ tự bit ngẫu nhiên)


## Đáp án Bài 6 — đếm hoán vị tối ưu = tích giai thừa kích thước nhóm

Theo invariant mục 13, tập hoán vị tối ưu chính là tập hoán vị tôn trọng dãy nhóm cuối cùng. Mỗi nhóm kích thước `s` xếp được `s!` cách, các nhóm độc lập → nhân lại.

In [39]:
from math import factorial, prod
from itertools import permutations
import random

def answer_count_optimal(nums):
    groups = [list(nums)]
    for i in range(15):
        b = 14 - i
        t = 0
        while t < len(groups) and all((x >> b) & 1 for x in groups[t]):
            t += 1
        if t == len(groups):
            continue
        has = [x for x in groups[t] if (x >> b) & 1]
        no = [x for x in groups[t] if not (x >> b) & 1]
        if has:
            groups[t:t+1] = [has, no]
    return prod(factorial(len(g)) for g in groups)

def count_optimal_bruteforce(nums):
    best = largest_power_bruteforce(nums)
    n = len(nums)
    return sum(1 for idx in permutations(range(n)) if power_of([nums[k] for k in idx]) == best)

assert answer_count_optimal([3, 11, 6, 2, 7]) == 4
assert answer_count_optimal([5, 5, 5]) == 6
assert answer_count_optimal([9, 6, 3]) == 1
rng = random.Random(6)
for _ in range(80):
    nums = [rng.randrange(16) for _ in range(rng.randint(1, 5))]
    assert answer_count_optimal(nums) == count_optimal_bruteforce(nums), nums
print("Bài 6: PASS | ví dụ n = 12 (4 giá trị, mỗi giá trị 3 lần):", answer_count_optimal([32767 >> (k % 4) for k in range(12)]))

Bài 6: PASS | ví dụ n = 12 (4 giá trị, mỗi giá trị 3 lần): 1296


# Tóm tắt — điều cần nhớ

## Hình ảnh
> Xếp `perm` thành **lưới bit**; `power[i]` = số chữ `1` liên tiếp từ trên xuống ở cột ứng với **bit `14 - i`**.

## Ba bước tư duy
1. **Brute force** (hoán vị) để hiểu đề và làm trọng tài.
2. **Tập thay cho thứ tự**: prefix AND chỉ phụ thuộc tập → DP bitmask; lexicographic → số cơ số `n + 1`.
3. **Greedy lọc dần theo bit cao → thấp**: tập lời giải tối ưu luôn là "mọi hoán vị tôn trọng một dãy nhóm"; mỗi bit chỉ **tách nhóm biên** thành `[có bit, không có bit]`.

## Template ngắn (Cách 3)

```python
groups, power = [list(nums)], [0] * 15
for i in range(15):
    b = 14 - i
    prefix, t = 0, 0
    while t < len(groups) and all((x >> b) & 1 for x in groups[t]):
        prefix += len(groups[t]); t += 1
    if t == len(groups):
        power[i] = len(nums); continue
    has = [x for x in groups[t] if (x >> b) & 1]
    no  = [x for x in groups[t] if not (x >> b) & 1]
    power[i] = prefix + len(has)
    if has:
        groups[t:t+1] = [has, no]
return power
```

## Độ phức tạp
```text
Cách 3: Time O(15 · n),     Space O(n)
Cách 4: Time O(n + 15 · D), Space O(D)
```

# Checklist trước khi submit

- [ ] Dùng đúng **bit `14 - i`** cho `power[i]` (không lật ngược)?
- [ ] Kiểm tra bit có **ngoặc đầy đủ**: `(x >> b) & 1` hoặc `x & (1 << b) != 0`?
- [ ] Vòng `while` **đi tiếp qua các nhóm "toàn có bit"**, không chỉ nhìn nhóm đầu?
- [ ] Có nhánh **`t == len(groups)` → `power[i] = n`** (tránh `IndexError`)?
- [ ] Chỉ tách nhóm khi `has` **khác rỗng**?
- [ ] **Không** dùng lối tắt "đếm số phần tử có bit" hay "sắp xếp giảm dần"?
- [ ] Trả về list **đủ 15 phần tử**, kể cả khi `nums` toàn số 0?
- [ ] Đã test hai ví dụ của đề + `[9, 6, 3]` (bẫy giảm dần) + `[3, 11, 6, 2, 7]` + `n = 1`?
- [ ] Đã đổi tên method theo đúng skeleton LeetCode hiển thị (ở đây giả định `lexLargestPower`)?